# Bronze Layer: Extract NSE API Data
This notebook extracts raw stock market data from the NSE API, flattens the JSON response, applies governance metadata, and writes the result into a Bronze Delta table in Azure Databricks.

In [0]:
import json
import logging
import time
import requests
import pandas as pd
import yaml
from datetime import datetime, timezone
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, to_date

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("bronze_extract")

## Load Configuration
Load Bronze layer configuration from `bronze_config.yaml`, including API settings, target storage path, and governance parameters.

In [0]:
# Determine config path relative to notebook location in Databricks
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    config_path = Path(notebook_path).parent.parent / "configs" / "bronze_config.yaml"
    # Add /Workspace prefix for file operations in Databricks
    config_path = "/Workspace" + str(config_path)
except Exception:
    config_path = "/Workspace/Shared/Stocksight/Extract_Transform_Load/bronze/configs/bronze_config.yaml"

with open(str(config_path), "r") as f:
    config = yaml.safe_load(f)

api_config = config["api"]
storage_config = config["storage"]
governance_config = config["governance"]

bronze_path = storage_config["bronze_path"]
bronze_format = storage_config.get("format", "delta")
partition_col = storage_config.get("partition_by", "ingestion_date")

logger.info(f"Loaded config from {config_path}")
logger.info(f"Bronze target path: {bronze_path}")
logger.info(f"Bronze format: {bronze_format}")

2026-05-02 15:47:25,481 - INFO - Loaded config from /Workspace/Shared/Stocksight/Extract_Transform_Load/bronze/configs/bronze_config.yaml
2026-05-02 15:47:25,485 - INFO - Bronze target path: /mnt/stockdata/bronze/nse_stock_raw
2026-05-02 15:47:25,486 - INFO - Bronze format: delta


## Initialize Spark Session
Create a Spark session configured for Databricks and Delta Lake.

In [0]:
spark = SparkSession.builder \
    .appName("Bronze_NSE_Extraction") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# Note: On serverless compute, spark.databricks.delta.schema.autoMerge.enabled is not available
# Use write options instead: .option("mergeSchema", "true") or .option("overwriteSchema", "true")
# spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")  # Not supported on serverless

logger.info("Spark session initialized successfully")

2026-05-02 15:47:25,713 - INFO - Received command c on object id p0
2026-05-02 15:47:25,729 - INFO - Spark session initialized successfully


## Extract NSE API Data
Call the NSE API with retries, normalize the JSON from the source, and preview the raw response.

In [0]:
def extract_nse_data(url: str, params: dict, timeout: int, retries: int, retry_delay: int) -> dict:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://www.nseindia.com/market-data/live-equity-market",
    })

    attempt = 0
    while attempt <= retries:
        try:
            logger.info(f"Calling NSE API (attempt {attempt + 1}/{retries + 1})")
            response = session.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except Exception as err:
            logger.warning(f"NSE API call failed: {err}")
            attempt += 1
            if attempt > retries:
                raise
            time.sleep(retry_delay)

raw_response = extract_nse_data(
    api_config["url"],
    api_config.get("params", {}),
    api_config.get("timeout", 15),
    api_config.get("retries", 3),
    api_config.get("retry_delay", 5)
)

raw_df = pd.json_normalize(raw_response["data"], sep="_")
raw_df.head(5)

2026-05-02 15:47:26,012 - INFO - Calling NSE API (attempt 1/4)


,priority,symbol,identifier,open,dayHigh,dayLow,lastPrice,previousClose,change,pChange,ffmc,yearHigh,yearLow,totalTradedVolume,stockIndClosePrice,totalTradedValue,lastUpdateTime,nearWKH,nearWKL,perChange365d,perChange30d,date365dAgo,date30dAgo,chartTodayPath,chart30dPath,chart365dPath,series,meta_symbol,meta_companyName,meta_industry,meta_activeSeries,meta_debtSeries,meta_isFNOSec,meta_isCASec,meta_isSLBSec,meta_isDebtSec,meta_isSuspended,meta_tempSuspendedSeries,meta_isETFSec,meta_isDelisted,meta_isin,meta_slb_isin,meta_listingDate,meta_isMunicipalBond,meta_isHybridSymbol,meta_segment,meta_quotepreopenstatus_equityTime,meta_quotepreopenstatus_preOpenTime,meta_quotepreopenstatus_QuotePreOpenFlag
0,1,NIFTY TOTAL MARKET,NIFTY TOTAL MARKET,12790.70,12802.80,12656.05,12761.50,12864.60,-103.10,-0.80,2.042177e+09,13842.6,6252.10,3469710725,0,1.280287e+12,30-Apr-2026 16:00:00,7.809949,-104.115417,3.21,8.64,29-Apr-2025,30-Mar-2026,https://nsearchives.nseindia.com/today/NIFTY-T...,https://nsearchives.nseindia.com/30d/NIFTY-TOT...,https://nsearchives.nseindia.com/365d/NIFTY-TO...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,RELIANCE,RELIANCEEQN,1409.00,1437.00,1393.10,1436.00,1425.40,10.60,0.74,9.672114e+12,1611.8,1290.00,30957881,0,4.384936e+10,NaN,10.907060,-11.317829,2.21,4.88,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/RELIANC...,https://nsearchives.nseindia.com/30d/RELIANCE-...,https://nsearchives.nseindia.com/365d/RELIANCE...,EQ,RELIANCE,Reliance Industries Limited,Refineries & Marketing,"[EQ, T0]",[],True,False,True,False,False,[],False,False,INE002A01018,INE002A01018,1995-11-29,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False
2,0,HDFCBANK,HDFCBANKEQN,770.00,778.80,762.25,774.75,779.00,-4.25,-0.55,1.182633e+13,1020.5,726.65,47998755,0,3.697632e+10,NaN,24.081333,-6.619418,-19.51,4.38,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/HDFCBAN...,https://nsearchives.nseindia.com/30d/HDFCBANK-...,https://nsearchives.nseindia.com/365d/HDFCBANK...,EQ,HDFCBANK,HDFC Bank Limited,Private Sector Bank,"[EQ, T0]",[],True,False,True,True,False,"[W3, IL]",False,False,INE040A01034,INE040A01034,1995-11-08,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False
3,0,SYNGENE,SYNGENEEQN,431.95,518.55,429.90,468.05,432.15,35.90,8.31,8.856553e+10,728.6,380.00,59274654,0,2.861010e+10,NaN,35.760362,-23.171053,-26.18,18.08,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/SYNGENE...,https://nsearchives.nseindia.com/30d/SYNGENE-E...,https://nsearchives.nseindia.com/365d/SYNGENE-...,EQ,SYNGENE,Syngene International Limited,Healthcare Research Analytics & Technology,"[EQ, T0]",[],False,False,True,False,False,[],False,False,INE398R01022,INE398R01022,2015-08-11,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False
4,0,MEESHO,MEESHOEQN,173.38,196.62,173.06,194.00,172.68,21.32,12.35,5.515687e+10,254.4,125.56,150163771,0,2.858217e+10,NaN,23.742138,-54.507805,0.00,31.64,None,01-Apr-2026,https://nsearchives.nseindia.com/today/MEESHOE...,https://nsearchives.nseindia.com/30d/MEESHO-EQ...,None,EQ,MEESHO,Meesho Limited,E-Retail/ E-Commerce,[EQ],[],False,False,True,False,False,[],False,False,INE0VDM01015,INE0VDM01015,2025-12-10,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False


## Apply Governance Metadata
Add ingestion metadata to the raw dataset before writing it into the Bronze Delta table.

In [0]:
from datetime import timezone

raw_df["ingestion_timestamp"] = pd.Timestamp(datetime.now(timezone.utc))
raw_df["ingestion_date"] = raw_df["ingestion_timestamp"].dt.date
raw_df["pipeline_run_id"] = governance_config.get("pipeline_run_id", datetime.now().strftime("%Y%m%d_run01"))
raw_df["source_system"] = governance_config.get("source_system", "NSE")
raw_df["file_name"] = f"nse_raw_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

raw_df.head(5)

2026-05-02 15:47:26,726 - INFO - Received command c on object id p0


,priority,symbol,identifier,open,dayHigh,dayLow,lastPrice,previousClose,change,pChange,ffmc,yearHigh,yearLow,totalTradedVolume,stockIndClosePrice,totalTradedValue,lastUpdateTime,nearWKH,nearWKL,perChange365d,perChange30d,date365dAgo,date30dAgo,chartTodayPath,chart30dPath,chart365dPath,series,meta_symbol,meta_companyName,meta_industry,meta_activeSeries,meta_debtSeries,meta_isFNOSec,meta_isCASec,meta_isSLBSec,meta_isDebtSec,meta_isSuspended,meta_tempSuspendedSeries,meta_isETFSec,meta_isDelisted,meta_isin,meta_slb_isin,meta_listingDate,meta_isMunicipalBond,meta_isHybridSymbol,meta_segment,meta_quotepreopenstatus_equityTime,meta_quotepreopenstatus_preOpenTime,meta_quotepreopenstatus_QuotePreOpenFlag,ingestion_timestamp,ingestion_date,pipeline_run_id,source_system,file_name
0,1,NIFTY TOTAL MARKET,NIFTY TOTAL MARKET,12790.70,12802.80,12656.05,12761.50,12864.60,-103.10,-0.80,2.042177e+09,13842.6,6252.10,3469710725,0,1.280287e+12,30-Apr-2026 16:00:00,7.809949,-104.115417,3.21,8.64,29-Apr-2025,30-Mar-2026,https://nsearchives.nseindia.com/today/NIFTY-T...,https://nsearchives.nseindia.com/30d/NIFTY-TOT...,https://nsearchives.nseindia.com/365d/NIFTY-TO...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-02 15:47:26.749260+00:00,2026-05-02,20260428_run01,NSE,nse_raw_20260502_154726.json
1,0,RELIANCE,RELIANCEEQN,1409.00,1437.00,1393.10,1436.00,1425.40,10.60,0.74,9.672114e+12,1611.8,1290.00,30957881,0,4.384936e+10,NaN,10.907060,-11.317829,2.21,4.88,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/RELIANC...,https://nsearchives.nseindia.com/30d/RELIANCE-...,https://nsearchives.nseindia.com/365d/RELIANCE...,EQ,RELIANCE,Reliance Industries Limited,Refineries & Marketing,"[EQ, T0]",[],True,False,True,False,False,[],False,False,INE002A01018,INE002A01018,1995-11-29,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False,2026-05-02 15:47:26.749260+00:00,2026-05-02,20260428_run01,NSE,nse_raw_20260502_154726.json
2,0,HDFCBANK,HDFCBANKEQN,770.00,778.80,762.25,774.75,779.00,-4.25,-0.55,1.182633e+13,1020.5,726.65,47998755,0,3.697632e+10,NaN,24.081333,-6.619418,-19.51,4.38,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/HDFCBAN...,https://nsearchives.nseindia.com/30d/HDFCBANK-...,https://nsearchives.nseindia.com/365d/HDFCBANK...,EQ,HDFCBANK,HDFC Bank Limited,Private Sector Bank,"[EQ, T0]",[],True,False,True,True,False,"[W3, IL]",False,False,INE040A01034,INE040A01034,1995-11-08,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False,2026-05-02 15:47:26.749260+00:00,2026-05-02,20260428_run01,NSE,nse_raw_20260502_154726.json
3,0,SYNGENE,SYNGENEEQN,431.95,518.55,429.90,468.05,432.15,35.90,8.31,8.856553e+10,728.6,380.00,59274654,0,2.861010e+10,NaN,35.760362,-23.171053,-26.18,18.08,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/SYNGENE...,https://nsearchives.nseindia.com/30d/SYNGENE-E...,https://nsearchives.nseindia.com/365d/SYNGENE-...,EQ,SYNGENE,Syngene International Limited,Healthcare Research Analytics & Technology,"[EQ, T0]",[],False,False,True,False,False,[],False,False,INE398R01022,INE398R01022,2015-08-11,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False,2026-05-02 15:47:26.749260+00:00,2026-05-02,20260428_run01,NSE,nse_raw_20260502_154726.json
4,0,MEESHO,MEESHOEQN,173.38,196.62,173.06,194.00,172.68,21.32,12.35,5.515687e+10,254.4,125.56,150163771,0,2.858217e+10,NaN,23.742138,-54.507805,0.00,31.64,None,01-Apr-2026,https://nsearchives.nseindia.com/today/MEESHOE...,https://nsearchives.nseindia.com/30d/MEESHO-EQ...,None,EQ,MEESHO,Meesho Limited,E-Retail/ E-Commerce,[EQ],[],False,False,True,False,False,[],False,False,INE0VDM01015,INE0VDM01015,2025-12-10,False,False,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,False,2026-05-02 15:47:26.749260+00:00,2026-05-02,20260428_run01,NSE,nse_raw_20260502_154726.json


### Mount ADLS Gen2 using Storage Account Key

In [0]:
# IMPORTANT: On serverless compute, direct ADLS Gen2 access requires one of:
# 1. Unity Catalog External Location (recommended)
# 2. Service Principal configured at workspace level
# 3. Managed Identity configured at workspace level
#
# Storage account keys cannot be set via spark.conf.set on serverless.
# Configure access via workspace admin settings before running this notebook.

# Original code (not supported on serverless):
# storage_key = dbutils.secrets.get(scope="my-scope", key="az-storage-key")
# spark.conf.set(
#     "fs.azure.account.key.adlsstock.dfs.core.windows.net",
#     storage_key
# )


# Test access (will fail if storage is not configured at workspace level)
dbutils.fs.ls("abfss://nsestock@adlsstock.dfs.core.windows.net/")
logger.info("ADLS Gen2 connected successfully")

2026-05-02 15:47:28,038 - INFO - ADLS Gen2 connected successfully


## Write to Bronze Layer
Convert the normalized pandas DataFrame to a Spark DataFrame and write it into the Bronze Delta path with partitioning.

In [0]:
bronze_df = spark.createDataFrame(raw_df)
bronze_df = bronze_df.withColumn(partition_col, to_date(current_timestamp()))

# Step 1: Write as Parquet to ADLS Gen2
adls_parquet_path = "abfss://nsestock@adlsstock.dfs.core.windows.net/bronze/nse_stock_details_parquet"

logger.info(f"Writing Parquet files to ADLS Gen2: {adls_parquet_path}")
bronze_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy(partition_col) \
    .option("compression", "snappy") \
    .save(adls_parquet_path)

logger.info(f"Successfully written Parquet files to {adls_parquet_path}")

# Verify the files were created
logger.info("Verifying files in ADLS Gen2...")
files = dbutils.fs.ls(adls_parquet_path)
logger.info(f"Found {len(files)} items in {adls_parquet_path}")

# Step 2: Read Parquet from ADLS Gen2
logger.info(f"Reading Parquet files from ADLS Gen2: {adls_parquet_path}")
parquet_df = spark.read.format("parquet").load(adls_parquet_path)

logger.info(f"Read {parquet_df.count()} rows from Parquet")

# Step 3: Write to Databricks managed Delta table in Unity Catalog
# Switch to the stocksight catalog
spark.sql("USE CATALOG dw_stocksight")
logger.info("Using catalog: dw_stocksight")

# Create schema
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_stocks_nse")

# Fully qualified table name: catalog.schema.table
table_name = "dw_stocksight.bronze_stocks_nse.nse_stock_details"

logger.info(f"Writing to managed Delta table: {table_name}")
parquet_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

logger.info(f"Successfully created managed table: {table_name}")

# Display sample data
display(spark.table(table_name).limit(5))

2026-05-02 15:47:28,603 - INFO - Writing Parquet files to ADLS Gen2: abfss://nsestock@adlsstock.dfs.core.windows.net/bronze/nse_stock_details_parquet
2026-05-02 15:47:29,864 - INFO - Successfully written Parquet files to abfss://nsestock@adlsstock.dfs.core.windows.net/bronze/nse_stock_details_parquet
2026-05-02 15:47:29,865 - INFO - Verifying files in ADLS Gen2...
2026-05-02 15:47:30,077 - INFO - Found 2 items in abfss://nsestock@adlsstock.dfs.core.windows.net/bronze/nse_stock_details_parquet
2026-05-02 15:47:30,078 - INFO - Reading Parquet files from ADLS Gen2: abfss://nsestock@adlsstock.dfs.core.windows.net/bronze/nse_stock_details_parquet
2026-05-02 15:47:30,544 - INFO - Read 750 rows from Parquet
2026-05-02 15:47:30,707 - INFO - Using catalog: dw_stocksight
2026-05-02 15:47:31,058 - INFO - Writing to managed Delta table: dw_stocksight.bronze_stocks_nse.nse_stock_details
2026-05-02 15:47:34,623 - INFO - Successfully created managed table: dw_stocksight.bronze_stocks_nse.nse_stock_de

priority,symbol,identifier,open,dayHigh,dayLow,lastPrice,previousClose,change,pChange,ffmc,yearHigh,yearLow,totalTradedVolume,stockIndClosePrice,totalTradedValue,lastUpdateTime,nearWKH,nearWKL,perChange365d,perChange30d,date365dAgo,date30dAgo,chartTodayPath,chart30dPath,chart365dPath,series,meta_symbol,meta_companyName,meta_industry,meta_activeSeries,meta_debtSeries,meta_isFNOSec,meta_isCASec,meta_isSLBSec,meta_isDebtSec,meta_isSuspended,meta_tempSuspendedSeries,meta_isETFSec,meta_isDelisted,meta_isin,meta_slb_isin,meta_listingDate,meta_isMunicipalBond,meta_isHybridSymbol,meta_segment,meta_quotepreopenstatus_equityTime,meta_quotepreopenstatus_preOpenTime,meta_quotepreopenstatus_QuotePreOpenFlag,ingestion_timestamp,pipeline_run_id,source_system,file_name,ingestion_date
1,NIFTY TOTAL MARKET,NIFTY TOTAL MARKET,12790.7,12802.8,12656.05,12761.5,12864.6,-103.10000000000036,-0.8,2.04217742363E9,13842.6,6252.1,3469710725,0,1.28028689671287E12,30-Apr-2026 16:00:00,7.809948998020605,-104.11541721981413,3.21,8.64,29-Apr-2025,30-Mar-2026,https://nsearchives.nseindia.com/today/NIFTY-TOTAL-MKT.svg,https://nsearchives.nseindia.com/30d/NIFTY-TOTAL-MKT.svg,https://nsearchives.nseindia.com/365d/NIFTY-TOTAL-MKT.svg,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-05-02T15:47:26.749Z,20260428_run01,NSE,nse_raw_20260502_154726.json,2026-05-02
0,RELIANCE,RELIANCEEQN,1409.0,1437.0,1393.1,1436.0,1425.4,10.6,0.74,9.67211448619389E12,1611.8,1290.0,30957881,0,4.384936180602E10,null,10.907060429333661,-11.317829457364342,2.21,4.88,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/RELIANCEEQN.svg,https://nsearchives.nseindia.com/30d/RELIANCE-EQ.svg,https://nsearchives.nseindia.com/365d/RELIANCE-EQ.svg,EQ,RELIANCE,Reliance Industries Limited,Refineries & Marketing,"List(EQ, T0)",List(),true,false,true,false,false,List(),false,false,INE002A01018,INE002A01018,1995-11-29,false,false,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,false,2026-05-02T15:47:26.749Z,20260428_run01,NSE,nse_raw_20260502_154726.json,2026-05-02
0,HDFCBANK,HDFCBANKEQN,770.0,778.8,762.25,774.75,779.0,-4.25,-0.55,1.182633365115668E13,1020.5,726.65,47998755,0,3.69763209018E10,null,24.081332680058793,-6.619417876556804,-19.51,4.38,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/HDFCBANKEQN.svg,https://nsearchives.nseindia.com/30d/HDFCBANK-EQ.svg,https://nsearchives.nseindia.com/365d/HDFCBANK-EQ.svg,EQ,HDFCBANK,HDFC Bank Limited,Private Sector Bank,"List(EQ, T0)",List(),true,false,true,true,false,"List(W3, IL)",false,false,INE040A01034,INE040A01034,1995-11-08,false,false,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,false,2026-05-02T15:47:26.749Z,20260428_run01,NSE,nse_raw_20260502_154726.json,2026-05-02
0,SYNGENE,SYNGENEEQN,431.95,518.55,429.9,468.05,432.15,35.9,8.31,8.856552856034E10,728.6,380.0,59274654,0,2.861009724618E10,null,35.76036233873182,-23.171052631578952,-26.18,18.08,30-Apr-2025,01-Apr-2026,https://nsearchives.nseindia.com/today/SYNGENEEQN.svg,https://nsearchives.nseindia.com/30d/SYNGENE-EQ.svg,https://nsearchives.nseindia.com/365d/SYNGENE-EQ.svg,EQ,SYNGENE,Syngene International Limited,Healthcare Research Analytics & Technology,"List(EQ, T0)",List(),false,false,true,false,false,List(),false,false,INE398R01022,INE398R01022,2015-08-11,false,false,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 09:07:37,false,2026-05-02T15:47:26.749Z,20260428_run01,NSE,nse_raw_20260502_154726.json,2026-05-02
0,MEESHO,MEESHOEQN,173.38,196.62,173.06,194.0,172.68,21.32,12.35,5.515687272191E10,254.4,125.56,150163771,0,2.858217217214E10,null,23.742138364779876,-54.50780503345014,0.0,31.64,null,01-Apr-2026,https://nsearchives.nseindia.com/today/MEESHOEQN.svg,https://nsearchives.nseindia.com/30d/MEESHO-EQ.svg,null,EQ,MEESHO,Meesho Limited,E-Retail/ E-Commerce,List(EQ),List(),false,false,true,false,false,List(),false,false,INE0VDM01015,INE0VDM01015,2025-12-10,false,false,EQUITY,30-Apr-2026 16:00:00,30-Apr-2026 